<br>

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:15%; float:right">

# Tutorial version.link und historisiertes Gemeindeverzeichnis

## Einführung

Dieses Tutorial ist dazu gedacht, eine Einführung in zwei verschiedene, aber zusammenhängende Themen zu geben:

1. das **historisierte Gemeindeverzeichnis**
2. das **version.link Schema**

Der Grund für dies Verknüpfung ist, dass das historisierte Gemeindeverzeichnis das version.link Schema verwendet. Dieses Tutorial zeigt dabei sowohl die grundlegenden Mechanismen des version.link Schemas auf, als auch die Arbeit mit dem historisierten Gemeindeverzeichnis. Das Tutorial ist also von Interesse für Personen, die sich über version.link informieren möchten - und das passiert hier anhand des historisierten Gemeindeverzeichnisses, aber auch für Personen, die sich primär für das historisierte Gemeindeverzeichnis interessieren, dafür aber über grundlegende Kenntnisse von version.link verfügen müssen.

## version.link Schema

Das version.link Schema wurde dazu entworfen, Daten, die in einem hierarchischen Zusammenhang stehen inklusive deren zeitlichen Entwicklung zu modellieren. Die Dokumentation von version.link ist unter https://version.link zu finden.

## Das historisierte Gemeindeverzeichnis

Gemeinden in der Schweiz unterliegen Veränderungen. Aktuell sind das primär Fusionen zwischen einzelnen Gemeinden. Das historisierte Gemeindeverzeichnis gibt sowohl einen Überblick über alle aktuell existierenden Gemeinden als auch über Gemeinden im Verlaufe der Zeit. Im historischen Gemeindeverzeichnis sind also auch solche Gemeinden zu finden, die heute nicht mehr existieren, weil sie fusioniert haben.

## Zielpublikum

Das Zielpublikum dieses Tutorials sind Personen, die über gute Grundkenntnisse in der Informatik verfügen, die aber noch wenig Knowhow zum Thema Linked Data, SPARQL und version.link besitzen.

## Interaktives Notebook

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

**Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
**Einzelne ausgewählte Zellen können sie danach abändern und mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Danach folgt eine [Kürzest-Einführung in Linked Data](#Linked-Data-Einführung). Danach folgt das eigentliche [Tutorial](#Tutorial).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Dass eine Zelle noch in Ausführung ist, ist am `[*]` links neben der Zelle erkennbar. Nach Abschluss der Ausführung erscheint statt eines `*` eine Zahl. Vor der ersten Ausführung ist eine leere Klammer `[]` zu sehen. Nachfolgende, wiederholte Ausführungen von Zellen werden aufgrund der gespeicherten Daten in Ihrem Browser viel schneller sein. 

**Speicherung von Änderungen**
Wenn Sie Änderungen am Tutorial vornehmen, werden diese Änderungen beim Sichern des Notebooks über File -> Save Notebook lokal in den Browser-Daten gespeichert. Bei einem späteren Öffnen des Notebooks werden wiederum die Änderungen aus den Browser-Daten geladen. Um wieder zur Ursprüngsversion des Tutorials zurückzukehren, gibt es die Möglichkeit, Ihr geändertes Notebook im File Browser (in der linken Seitenleiste das Ordner-Symbol) umzubenennen. Damit wird automatisch das ursprüngliche Notebook unter dem ursprünglichen Namen wieder neu vom Server geladen. Das gleiche passiert auch, wenn Sie das Notebook im File Browser einfach löschen. Ein Arbeiten in einem Chrome-Inkognito Fenster verhindert, dass Änderungen überhaupt persistent gespeichert werden.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [ ]:
%pip install -q ipywidgets==8.0.4 ipycytoscape networkx nbformat plotly
import ipycytoscape as cy
import networkx as nx
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from ext.sparql import query, display_result

# Linked Data Einführung

Linked Data beschreibt ein **Framework für den Umgang mit Daten**, die sowohl für Menschen nützlich sein sollen, als auch maschinenlesbar sind inkl. einer von Computern verarbeitbaren Semantik. Also sowohl Menschen als auch Computer sollen die Daten "verstehen" und interpretieren können. 

## RDF

Das für Linked Data verwendete Datenformat ist RDF (Resource Description Framework). Das bedeutet, dass die Daten nicht als Tabellen (wie beispielsweise in relationalen Datenbanken) sondern als **Triples** abgespeichert werden. Triples folgen der grammatikalischen Struktur **Subjekt -> Prädikat -> Objekt** und können auch als grammatikalischer Satz verstanden werden. 

Die Information "**Der Apfel ist grün**" wird also mit dem Tripel **Apfel -> ist -> grün** ausgedrückt. Alle Teile eines Triples sind dabei durch weitere Eigenschaften definiert und beschrieben die wiederum in Form von Triples beschrieben sind. Diese vielseitigen Verknüpfungen führen zu einer Netzwerkstruktur, zu einem sogenannten Graphen.

Nachfolgendes Bild aus dem [W3C Primer für RDF](https://www.w3.org/TR/rdf11-primer/) veranschaulicht diese Zusammenhänge:

![](https://www.w3.org/TR/rdf11-primer/example-graph.jpg)

## URI

Eine weitere wichtige Eigenschaft von Linked Data ist, dass alle Teile eines Triples, also Subjekt, Prädikat und Objekt weltweit eindeutig identifizierbar sind. Dafür werden URI (Universal Resource Identifier) eingesetzt. Die URI *xy* beispielsweise ist der weltweit eindeutige Identifier für die Stadt Bern. Typischerweise lassen sich URI **dereferenzieren**, das heisst, ein Request auf die entsprechende URI (bspw. in dem man sie in die Adresszeile eines Browsers eintippt) führt zu einer Website, die Infos zur entsprechenden URI enthält. Im Falle der URI der Stadt Bern wird man auf eine Webpage weitergeleitet, die diverse Informationen zur Stadt Bern enthält.

## Weitere Informationen zu Linked Data

Wer vertieft in das Thema Linked Data einsteigen möchte, dem sei beispielsweise [diese Youtube Playlist](https://www.youtube.com/watch?v=ON0wf0SEPx8&list=PLoOmvuyo5UAfY6jb46jCpMoqb-dbVewxg) empfohlen.

## SPARQL

SPARQL ist eine Query-Sprache für Linked Data Triple Stores. Für eine allgemeine Einführung in SPARQL siehe z.B.: https://jena.apache.org/tutorials/sparql.html

Abfragen (engl. Queries) können entweder direkt über das [SPARQL Web-Interface von Fedlex](https://fedlex.data.admin.ch/de-CH/sparql) eingegeben und ausgeführt werden, oder als HTTP-POST Request an den SPARQL-Endpoint (https://fedlex.data.admin.ch/sparqlendpoint) gesendet werden.

Die letztere Methode erlaubt es, eigene Anwendungen zu bauen, die automatisch aktuelle Daten von Fedlex abfragen können. Für dieses Tutorial verwenden wir diese Methode. Die Abfragen sind jedoch in beiden Fällen identisch.

Um im Tutorial eine neue SPARQL Abfrage zu erstellen, erzeugen Sie eine neue Zelle für Code ("Plus-Zeichen" in der Titelzeile des aktuellen Tabs drücken und im Dropdown "Code" auswählen. Danach können sie mit dem Python Befehl `await query(query_string)` die Abfrage ausführen, welche als Resultat ein Pandas Dataframe zurückgibt, welches sinnvollerweise einer Variable zugewiesen wird. Das Schlüsselwort `await` ist notwendig, weil die Abfrage asynchronen Code enthält. Die Anzeige des Dataframes erfolgt sinnvollerweise mit dem Befehl `display_result(df)`, welcher daführ sorgt, dass die URI im Dataframe als klickbare Links dargestellt werden. Die dreifachen Anführungszeichen des `query_string` ermöglichen, den String über mehrere Zeilen umzubrechen und damit eine übersichtliche Darstellung der Query zu erreichen.

Soll eine Query aus dem Tutorial über das SPARQL Web-Interface ausgeführt werden, kopieren Sie einfach den Teil zwischen den `"""` und fügen sie im [SPARQL Web-Interface](https://ld.admin.ch/sparql) ein.

## Pattern Matching

SPARQL Queries sind Aufträge an den Computer, bestimme Muster (Pattern) in den Daten zu finden (matching). Es können also mit Hilfe von SPARQL Muster vorgegeben werden, und die Datenbank gibt alle Triples zurück, die dieses Muster erfüllen. Einzelne Positionen der Triples können dabei bei einer Abfrage bewusst undefiniert gelassen und mit einer Variable bezeichnet werden. Variablen beginnen mit `?` und werden bei der Abfrage durch alle möglichen Elemente gefüllt, die dieses Pattern erfüllen. Eine ausführlicher Anleitung zum Pattern Matching ist [hier](https://programminghistorian.org/en/lessons/retired/graph-databases-and-SPARQL#rdf-in-brief) zu finden. 

## Prefixes

Um immer wiederkehrende URI kompakter zu notieren, werden sogenannte Prefixes verwendet, im nachfolgenden werden folgende Prefixes benutzt:

```
PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
```

# Tutorial

## Identitäten und Versionen

Unter version.link sind die zentralen Entitäten **Identitäten** und **Versionen**, wobei die Identitäten die eigentlichen Objekte des Interesses darstellen. Hier bspw. die politischen Gemeinden der Schweiz. Diese Identitäten basieren auf einer bestimmten Version. Die Version wiederspiegeln also die zeitliche Entwicklung der Identität. Änderung am Objekt des Interesses (bspw. Namensänderung einer Gemeinde) ziehen eine neue Version nach sich, auf der, die Identität dann entsprechend basiert.

### Identität der Gemeinde "Bern"

In [ ]:
df = await query("""

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/351> ?p ?o.

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

Die entscheidenden Angaben sind dabei der Typ `vl:Identity`, die Verbindung zur Version, auf der die Identität basiert über den Link `vl:version` zu `https://ld.admin.ch/municipality/version/15029` und die Einordnung in der Hierarchie über `schema:isPartOf` zu `https://ld.admin.ch/district/246`. Die Hierarchie wird zusätzlich über `schema:containedInPlace` über alle föderalen Ebenen abgebildet.

### Version der Gemeinde "Bern"

In [ ]:
df = await query("""

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/version/15029> ?p ?o.

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

Die entscheidenden Angaben sind dabei der Typ `vl:Version`, die Verbindung zur Identität, auf der diese basiert über den Link `vl:identity` zu `https://ld.admin.ch/municipality/351` und die Einordnung in der Hierarchie über `schema:isPartOf` zu `https://ld.admin.ch/district/version/10288`.

## Informationen zu Gemeinde-Identitäten

Basierend auf den Identitäten können bereits verschiedene Informationen/Statistiken erstellt werden zum Zustand der "Gemeindelandschaft" der Schweiz:

### Anzahl aller Gemeinden

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

Die Suche nach allen Gemeinden über den Typ `admin:PoliticalMunicipality` ist so zu verstehen, dass alle Gemeinden zurückgeben werden, die aktuell existieren und je existiert haben. Gemeinden, die nicht mehr aktuell sind, erhalten zusätzlich ein Typ `vl:Deprecated`, nachdem zusätzlich gefragt werden kann:

### Anzahl aller nicht mehr aktuellen Gemeinden

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity;
        a vl:Deprecated.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

### Anzahl aller aktuellen Gemeinden

Wenn nur die aktuellen Gemeinden interessieren, dann muss danach gefiltert werden, dass kein `vl:Deprecated` existieren darf:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity.
    
    FILTER NOT EXISTS {?muni a vl:Deprecated}

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

### Anzahl Gemeinden pro Anfangsbuchstabe

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?char (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE 
{
    ?muni a admin:PoliticalMunicipality;
        a vl:Identity;
        schema:name ?name.

    BIND(SUBSTR(?name, 1, 1) AS ?char)

} GROUP BY ?char ORDER BY DESC(?count)

""", "https://test.lindas.admin.ch/query")

display_result(df)

### Gemeinden mit 'wil' im Namen

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE 
{
    ?muni a admin:PoliticalMunicipality;
        a vl:Identity;
        schema:name ?name.
        
    #FILTER(CONTAINS(?name, 'wil') || CONTAINS(?name, 'Wil'))
    FILTER REGEX(?name, 'wil', 'i').

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

### Anzahl aktueller Gemeinden pro Kanton

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?canton ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton.

            MINUS {?muni a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

Dieses Resultat kann nun mit Hilfe von Matplotlib direkt als Balkendiagramm dargestellt werden:

In [ ]:
plt.barh(df["cantonName"], df["numberOfMunies"])
plt.suptitle("Anzahl Gemeinden pro Kanton")
plt.xlabel("Anzahl Gemeinden")
plt.grid(color = "grey", linestyle = "--", linewidth=1)
plt.show()

### Durchschnittliche Grösse der aktuellen Gemeinden pro Kanton

Die nachfolgende Abfrage zeigt, wie Zusatzinformationen aus anderen Triple-Stores mit einer einzigen SPARQL Query eingebunden werden können (sog. "Federated-Query"). Dazu wird die `SERVICE` Konstruktion verwendet:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?cantonName ?muniMeanSize
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.
    
    SERVICE <https://geo.ld.admin.ch/query> {
    
        ?cantonGEO schema:about ?canton;
            purl:hasVersion ?version.
        ?version purl:issued "2023-01-01"^^xsd:date.
        ?version <http://dbpedia.org/property/area> ?area.
    
    }
    
    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton.

            MINUS {?muni a vl:Deprecated}

        } GROUP BY ?canton
    
    }
    
    BIND(ROUND(xsd:float(xsd:string(?area))/?numberOfMunies) AS ?muniMeanSize).

    
} ORDER BY DESC(?muniMeanSize)

""", "https://test.lindas.admin.ch/query")

display_result(df)

### Anzahl nicht mehr aktueller Gemeinden pro Kanton

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton;
                a vl:Deprecated.

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

Diese Abfrage zeigt, dass unterschiedliche Kantone das Fusionieren von Gemeinden sehr unterschiedlich stark fördern.

### Anzahl aktueller Bezirke pro Kanton 

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfDistricts
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?district) AS ?numberOfDistricts)
        WHERE {

            ?district a admin:District;
                a vl:Identity;
                schema:isPartOf ?canton.

            MINUS {?district a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfDistricts)

""", "https://test.lindas.admin.ch/query")

display_result(df)

## Zeitliche Entwicklung mit Hilfe von Gemeinde-Versionen

Wenn nicht nur der aktuelle Zustand von Interesse ist, sondern eine zeitliche Entwicklung oder ein bestimmter Zeitpunkt, muss mit den Versionen der Gemeinden gearbeitet werden:

### Gemeinden in einem Kanton zu einem bestimmten Zeitpunkt

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?date ?muniVersion ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    VALUES ?date {
        "2000-01-01"^^xsd:date
        "2023-01-01"^^xsd:date
    }
    
    ?muniVersion a vl:Version;
        vl:identity ?muni;
        schema:isPartOf/schema:isPartOf <https://ld.admin.ch/canton/8>;
        schema:validFrom ?start;
        schema:legalName ?name.
    ?muni a admin:PoliticalMunicipality.
        
    OPTIONAL {?muniVersion schema:validThrough ?stop.}
        
    FILTER (?date >= ?start)
    FILTER (!BOUND(?stop) || ?date <= ?stop)

}

""", "https://test.ld.admin.ch/query")

display_result(df)

### Verlauf der Anzahl Gemeinden in der Schweiz über die Zeit

In [ ]:
df = await query("""

    PREFIX vl: <https://version.link/>
    PREFIX admin: <https://schema.ld.admin.ch/>
    PREFIX schema: <http://schema.org/>
    PREFIX time: <http://www.w3.org/2006/time#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    SELECT ?date (COUNT(?version) as ?muniCount)
    FROM <https://lindas.admin.ch/fso/register>
    FROM <https://lindas.admin.ch/meta>
    WHERE {
    
        ?year schema:inDefinedTermSet <https://ld.admin.ch/time/year>;
            time:year ?year_int.
        BIND(xsd:date(CONCAT(STR(?year_int), "-01-01")) AS ?date)
        
        FILTER(?date < xsd:date(now()))

        ?version vl:identity ?identity;
            schema:validFrom ?start.
        ?identity a admin:Municipality.
        #?identity schema:containedInPlace <https://ld.admin.ch/canton/2>. # falls Einschränkung auf einen bestimmten Kanton gewünscht ist

        OPTIONAL {?version schema:validThrough ?stop.}

        FILTER (?date >= ?start)
        FILTER (!BOUND(?stop) || ?date <= ?stop)

    } GROUP BY ?date


    """, "https://test.ld.admin.ch/query")

df["date"] = pd.to_datetime(df["date"])
plt.plot(df["date"], df["muniCount"])
plt.suptitle("Verlauf der Anzahl Gemeinden in der Schweiz")
plt.xlabel("Datum")
plt.ylabel("Anzahl Gemeinden")
plt.grid(color = "grey", linestyle = "--", linewidth=1)
plt.show()

### Änderungsgeschichte einer bestimmten Gemeinde

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?participants ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        schema:name "Münsingen";
        vl:version ?final.
    ?participants vl:successor* ?final;
        schema:name ?name.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

Jetzt macht diese tabellarische Darstellung nur mässig Sinn, wichtiger wäre zu sehen, welche Gemeinde in welche übergegangen ist:

In [ ]:
edge_df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?sourceName ?target ?targetName
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        schema:name "Münsingen";
        vl:version ?final.
    ?source vl:successor* ?final;
        schema:name ?sourceName;
        vl:successor ?target.
    ?target schema:name ?targetName.
}


""", "https://test.lindas.admin.ch/query")


ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)

G = nx.from_pandas_edgelist(edge_df, source="source", target="target", create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(name)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

### Alle Gemeinden, die sich geteilt haben

Typischerweise fusionieren Gemeinden ja eher, also aus mehreren Gemeinden wird eine gebildet. Es gibt aber auch den anderen Weg, wo Gemeinden sich teilen:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?source schema:name ?name
    
    {
        SELECT ?source (COUNT(?target) as ?successors)
        WHERE {

            ?source vl:successor ?target;

            MINUS {?source a vl:ChangeEvent}
            
            FILTER(regex(str(?source), "municipality" ) )
        } 
        GROUP BY ?source
    }
    FILTER(?successors > 1) 
} ORDER BY DESC(?successors)

""", "https://test.lindas.admin.ch/query")

display_result(df)

### Versionen ohne Zeitausdehnung

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version a vl:Version;
        schema:name ?name;
        schema:validFrom ?start;
        schema:validThrough ?stop.
        
    FILTER(?start = ?stop)
    
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

## Change Events

Um die verschiedenen Ereignisse rund um die Entwicklung von Gemeinden genauer zu untersuchen, sind die `vl:ChangeEvent` interessant:

### Beispiel Change Event 

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT *
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/changeevent/3953> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

### Namensänderung von Gemdeinden

Folgende Gemeinden haben ihren Namen irgendwann geändert:

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?before ?after
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?version vl:endEvent ?event;
        schema:name ?before;
        vl:successor ?successor;
        vl:identity ?identity.
    ?identity a admin:Municipality.
    ?event a vl:ChangeOfName.
    ?successor schema:name ?after.
} 


""", "https://test.lindas.admin.ch/query")

display_result(df)

## eCH-0071 Daten abfragen

Das historisierte Gemeindeverzeichnis ist ursprünglich eine XML Datei, die nach dem [eCH-0071 Standard](https://www.ech.ch/de/ech/ech-0071) aufgebaut ist und [hier](https://www.bfs.admin.ch/asset/de/23886070) erhältlich ist. Ausserdem existiert eine sehr interessante [Erläuterung](https://www.bfs.admin.ch/asset/de/18484929) zu diesem Verzeichnis, das insbesondere auch die verwendeten Codes beinhaltet. Dieses XML Verzeichnis wurde ebenfalls nach Linked Data transferiert und mit den Daten gemäss version.link Standard verlinkt. Dies ermöglicht es, sehr detaillierte Angaben insbesondere bei Veränderungsprozessen zu erhalten:

### Detaillierte Infos über Veränderungsprozesse

In [ ]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX ech: <https://ld.admin.ch/ech/71/>

SELECT DISTINCT ?predecessor ?predecessorName ?abolitionMode ?admissionDate ?admissionMode ?successorName ?successor
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        schema:name "Klosters";
        vl:version ?final.
    ?predecessor vl:successor* ?final;
        schema:name ?predecessorName;
        vl:successor ?successor;
        prov:hadPrimarySource ?predecessorECH.
    ?successor prov:hadPrimarySource ?successorECH;
        schema:name ?successorName;
        schema:validFrom ?admissionDate.
        
    ?predecessorECH ech:municipalityAbolitionModeId ?abolitionMode.
    ?successorECH ech:municipalityAdmissionModeId ?admissionMode.

    MINUS {?predecessor a vl:ChangeEvent}
    

} ORDER BY ?admissionDate


""", "https://test.lindas.admin.ch/query")

display_result(df)

Hier kann man zeitlich folgende Entwicklung sehen:

- Auf 1.1.1974 wird die Gemeinde Klosters in Klosters-Serneus umbenannt (Serneus wurde übrigens 1872 eingemeindet!)
- Auf 1.1.2001 werden im Kanton Graubünden die Bezirke geändert, was für die Gemeinde Saas und Klosters-Serneus eine neue Version ergibt
- Auf 1.1.2016 fusionieren Saas und Klosters-Serneus. Für Saas ist es eine Aufhebung (29 zu 26), für Klosters-Serneus ist es eine Gebietsänderung (26 zu 26), zusammen spricht man von einer Eingemeindung
- Auf 1.1.2017 gibt es nochmals eine Bezirksreform
- Auf 1.1.2021 wird Klosters-Serneus wieder umbenannt nach Klosters

## Path Queries

Um die vollständige Entwicklung eines "Gemeinde-Clusters" darzustellen, genügen SPARQL Property-Paths nicht. Dafür sind Path Queries nötig, welche jedoch nicht Teil der SPARQL Spezifikation sind, sondern proprietär je nach Triple Store gehandhabt werden. Der von LINDAS verwendete Stardog Triple Store hat eine [Dokumentation zu Path Queries](https://docs.stardog.com/query-stardog/path-queries).

In [ ]:
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO
import json

async def path(query_string, address):
    
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "application/sparql-results+json" })),
        )
    except:
        raise RuntimeError("Fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        return res
    else:
        raise RuntimeError("Response status " + str(resp.status) + ":" + await resp.text())

### Änderungen von "Gemeinde-Clustern"

In [ ]:
res = await path("""

PREFIX schema: <http://schema.org/>

PATHS 
#START ?x = <https://ld.admin.ch/municipality/version/14062> #Berg (TG)
#START ?x = <https://ld.admin.ch/municipality/version/13445> #Rubigen
START ?x = <https://ld.admin.ch/municipality/version/14091> #Eschlikon
#START ?x = <https://ld.admin.ch/municipality/version/12612> #Lavertezzo
#START ?x = <https://ld.admin.ch/municipality/version/14091> #Eschlikon
END ?y 
VIA {

    ?x ?p ?y.
    ?x schema:name ?xName.
    ?y schema:name ?yName.
    
    FILTER(?p = <https://version.link/successor> || ?p = <https://version.link/predecessor>)

}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)

results = []

for entry in res["results"]["bindings"]:
    if "x" in entry:
        if entry["p"]["value"] == "https://version.link/successor":
            results.append(
                {
                    "source": entry["x"]["value"], 
                    "sourceName": entry["xName"]["value"],
                    "targetName": entry["yName"]["value"],
                    "target": entry["y"]["value"]
                }
            )
        else:
            results.append(
                {
                    "source": entry["y"]["value"], 
                    "sourceName": entry["yName"]["value"],
                    "targetName": entry["xName"]["value"],
                    "target": entry["x"]["value"]
                }
            )
            
edge_df = pd.DataFrame(results)

ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)

G = nx.from_pandas_edgelist(edge_df, source="source", target="target", create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '12px',
            'label': 'data(name)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

Hier ist gut zu sehen, dass eine SPARQL Query nach allen Vorgängern von der letzten Version einer Gemeinde nicht alle beteiligten Versionen gefunden hätte.

In [ ]:
res = await path("""

PREFIX schema: <http://schema.org/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX ech: <https://ld.admin.ch/ech/71/>

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14091>
END ?y 

VIA {

    ?x ?p ?y.
    
    ?x schema:name ?xName;
        prov:hadPrimarySource ?xECH;
        schema:validFrom ?xAdDate.
    ?y schema:name ?yName;
        prov:hadPrimarySource ?yECH;
        schema:validFrom ?yAdDate.
        
    OPTIONAL {?x schema:validThrough ?xAbDate.}
    OPTIONAL {?y schema:validThrough ?yAbDate.}
        
    OPTIONAL {?xECH ech:municipalityAbolitionModeId ?xECHAbId.}
    OPTIONAL {?xECH ech:municipalityAdmissionModeId ?xECHAdId.}
        
    OPTIONAL {?yECH ech:municipalityAbolitionModeId ?yECHAbId.}
    OPTIONAL {?yECH ech:municipalityAdmissionModeId ?yECHAdId.}
    
    FILTER(?p = <https://version.link/successor> || ?p = <https://version.link/predecessor>)

}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)

results = []

for entry in res["results"]["bindings"]:
    if "x" in entry:
        if entry["p"]["value"] == "https://version.link/successor":
            results.append(
                {
                    "source": entry["x"]["value"], 
                    "sourceName": entry["xName"]["value"],
                    "abolitionDate": entry["xAbDate"]["value"],
                    "abolitionId": entry["xECHAbId"]["value"],
                    "admissionId": entry["yECHAdId"]["value"],
                    "admissionDate": entry["yAdDate"]["value"],
                    "targetName": entry["yName"]["value"],
                    "target": entry["y"]["value"]
                }
            )
        else:
            results.append(
                {
                    "source": entry["y"]["value"], 
                    "sourceName": entry["yName"]["value"],
                    "abolitionDate": entry["yAbDate"]["value"],
                    "abolitionId": entry["yECHAbId"]["value"],
                    "admissionId": entry["xECHAdId"]["value"],
                    "admissionDate": entry["xAdDate"]["value"],
                    "targetName": entry["xName"]["value"],
                    "target": entry["x"]["value"]
                }
            )
            
df = pd.DataFrame(results).drop_duplicates()

df["abolitionDate"] = pd.to_datetime(df["abolitionDate"])
df["admissionDate"] = pd.to_datetime(df["admissionDate"])

            
df.sort_values("admissionDate", inplace=True)

display_result(df)

In dieser Darstellung kann man die Entwicklung gut verfolgen (insbesondere mit der zusätzlichen grafischen Darstellung von oben):

- Ende 1996 passieren verschiedene Dinge gleichzeitig:
    - die Gemeinde Wallenwil wird in die Gemeinde Eschlikon eingemeindet
    - die Gemeinden Wiezikon, Horben und Busswil (TG) werden in die Gemeinde Sirnach eingemeindet
    - die neuen Gemeinden Eschlikon und Sirnach tauschen noch Land ab
- Auf den 1.1.2011 findet eine Änderung der Bezirke statt

Somit ist auch klar, was es mit diesen Versionen auf sich hat, die das gleiche Datum für `schema:validFrom` und `valid:Through` haben. Dies ist nötig, wenn verschiedene Wechsel "gleichzeitig" passieren. In diesem Fall also, dass Gemeinden fusionieren und gleichzeitig noch mit einer anderen Gemeinde Land abtauschen.

In [ ]:
def color_map(code):
    if code == "29-26":
        return "blue"
    elif code == "26-26":
        return "green"
    else:
        return "black"

In [ ]:
res = await path("""

PREFIX schema: <http://schema.org/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX ech: <https://ld.admin.ch/ech/71/>

PATHS 
START ?x = <https://ld.admin.ch/municipality/version/14091>
END ?y 

VIA {

    ?x ?p ?y.
    
    ?x schema:name ?xName;
        prov:hadPrimarySource ?xECH;
        schema:validFrom ?xAdDate.
    ?y schema:name ?yName;
        prov:hadPrimarySource ?yECH;
        schema:validFrom ?yAdDate.
        
    OPTIONAL {?x schema:validThrough ?xAbDate.}
    OPTIONAL {?y schema:validThrough ?yAbDate.}
        
    OPTIONAL {?xECH ech:municipalityAbolitionModeId ?xECHAbId.}
    OPTIONAL {?xECH ech:municipalityAdmissionModeId ?xECHAdId.}
        
    OPTIONAL {?yECH ech:municipalityAbolitionModeId ?yECHAbId.}
    OPTIONAL {?yECH ech:municipalityAdmissionModeId ?yECHAdId.}
    
    FILTER(?p = <https://version.link/successor> || ?p = <https://version.link/predecessor>)

}


""", "https://test.lindas.admin.ch/query")

res = json.loads(res)

results = []

for entry in res["results"]["bindings"]:
    if "x" in entry:
        if entry["p"]["value"] == "https://version.link/successor":
            results.append(
                {
                    "source": entry["x"]["value"], 
                    "sourceName": entry["xName"]["value"],
                    "abolitionDate": entry["xAbDate"]["value"],
                    "abolitionId": entry["xECHAbId"]["value"],
                    "admissionId": entry["yECHAdId"]["value"],
                    "admissionDate": entry["yAdDate"]["value"],
                    "targetName": entry["yName"]["value"],
                    "target": entry["y"]["value"]
                }
            )
        else:
            results.append(
                {
                    "source": entry["y"]["value"], 
                    "sourceName": entry["yName"]["value"],
                    "abolitionDate": entry["yAbDate"]["value"],
                    "abolitionId": entry["yECHAbId"]["value"],
                    "admissionId": entry["xECHAdId"]["value"],
                    "admissionDate": entry["xAdDate"]["value"],
                    "targetName": entry["xName"]["value"],
                    "target": entry["x"]["value"]
                }
            )
            
df = pd.DataFrame(results).drop_duplicates()

df["abolitionDate"] = pd.to_datetime(df["abolitionDate"])
df["admissionDate"] = pd.to_datetime(df["admissionDate"])
df["changeMode"] = df["abolitionId"] + "-" + df["admissionId"]
df["color"] = df["changeMode"].map(color_map)

            
df.sort_values("admissionDate", inplace=True)

display_result(df)


edge_df = df

ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)

G = nx.from_pandas_edgelist(edge_df, source="source", target="target", edge_attr = "color", create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '12px',
            'label': 'data(name)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
            "line-color": 'data(color)'
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni